# Random Forest Classification — Google Colab

**Goal:** Predict a **binary class** (0/1) using **Random Forest Classification** — an **ensemble** of Decision Trees that **vote** to reduce overfitting and improve accuracy.

| Example | Feature(s) (X) | Target (y) | Dataset |
|---------|----------------|------------|---------|
| **Example 1** | Age, EstimatedSalary | Purchased (0/1) | `../Datasets/social_network_ads.csv` |
| **Example 2** | CreditScore, Income, LoanAmount, YearsEmployed | Approved (0/1) | `../Datasets/loan_approval.csv` |

| Phase | Topic | Cells |
|-------|-------|-------|
| Phase 0 | Setup | Install & Imports |
| — | Algorithm Guide | Bagging, bootstrap, majority vote |
| Phase 1 | Data Pre-processing | Load → Cleaning → Encoding → Split |
| Phase 2 | Algorithm | Train → Predict → Visualize → Evaluate |

> **Run:** Runtime → Run all (or Ctrl+F9)

---
# Algorithm Guide — Random Forest Classification

## What is Random Forest?

**Random Forest** = **many Decision Trees** trained on **different random subsets** of data and features, then their class predictions are combined by **majority vote**.

It is a **Bagging** (Bootstrap Aggregating) ensemble method invented by Leo Breiman (2001).

## How It Works (Step by Step)

| Step | Name | What Happens |
|------|------|--------------|
| 1 | **Bootstrap sample** | Draw n rows **with replacement** from training data |
| 2 | **Random features** | At each split, consider only a **random subset** of features |
| 3 | **Grow tree** | Build a CART classification tree on that bootstrap sample |
| 4 | **Repeat** | Create `n_estimators` trees (e.g. 100) |
| 5 | **Aggregate** | Final class = **majority vote** across all trees |

## Prediction Rule (Classification)

For a new sample **x**:

1. Each tree **b** predicts a class: `ŷ_b(x) ∈ {0, 1}`
2. Final prediction = class with the **most votes**
3. `predict_proba()` = fraction of trees voting for each class

**Example:** 100 trees → 72 vote "Yes", 28 vote "No" → predict **Yes**, P(Yes) = 0.72

## Key Hyperparameters

| Parameter | Role | Effect |
|-----------|------|--------|
| `n_estimators` | Number of trees in the forest | More trees → more stable (diminishing returns) |
| `max_depth` | Max depth of each tree | Limits overfitting per tree |
| `min_samples_split` | Min samples to split a node | Higher → simpler trees |
| `min_samples_leaf` | Min samples in a leaf | Higher → smoother decisions |
| `max_features` | Features considered per split | `'sqrt'` adds diversity between trees |
| `bootstrap` | Use bootstrap sampling | `True` (default) — core of bagging |
| `random_state` | Random seed | Reproducible forest |

## Random Forest vs Single Decision Tree (CART)

| | Single CART | Random Forest |
|---|-------------|---------------|
| Trees | **1** tree | **Many** trees (ensemble) |
| Overfitting | **High** risk if deep | **Lower** — voting reduces variance |
| Boundary | Axis-aligned steps | **Smoother** voted regions |
| Interpretability | Easy (`plot_tree`) | Use **feature importances** |
| Scaling | Not required | **Not required** |
| Speed | Fast train/predict | Slower (trains B trees) |

## Feature Importance

Random Forest averages how much each feature **reduces Gini impurity** across all splits in all trees. Higher value = more influential for classification.

## What the Student Must Remember

1. Random Forest = **Bagging + random feature subsets** at each split.
2. Final class = **majority vote** of all trees (classification).
3. **No feature scaling** needed — same as Decision Trees.
4. More trees (`n_estimators`) usually help until performance plateaus.
5. Use **feature importances** when the forest is too large to visualize.

## Phase 0 — Cell 0: Install Libraries

Google Colab usually includes most libraries. This cell ensures required packages are available.

**What this cell does:** Installs scikit-learn, pandas, matplotlib, numpy, and seaborn quietly.

In [ ]:
# Install required libraries quietly (-q hides output)
!pip install -q scikit-learn pandas matplotlib numpy seaborn

## Phase 0 — Cell 1: Import Libraries

Import libraries for preprocessing, Random Forest classification, and evaluation.

**What this cell does:** Loads numpy, pandas, matplotlib, sklearn RandomForestClassifier, and metrics.

In [ ]:
# --- Import libraries ---
import numpy as np              # Numerical operations and arrays
import pandas as pd             # Load and manipulate tabular data
import matplotlib.pyplot as plt # Create charts and plots
import seaborn as sns           # Statistical visualizations (heatmaps)

from sklearn.model_selection import train_test_split       # Split data into train/test
from sklearn.impute import SimpleImputer                   # Fill missing values
from sklearn.ensemble import RandomForestClassifier        # Random Forest ensemble classifier
from sklearn.metrics import (                              # Classification metrics
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

plt.rcParams['figure.figsize'] = (10, 6)  # Default plot size: width=10, height=6 inches
sns.set_theme(style='whitegrid')            # Clean white background with grid lines
np.random.seed(42)                          # Fix random seed for reproducible splits

print('Libraries ready')                      # Confirm all imports loaded successfully

---
# Example 1: Social Network Ads — Random Forest Classification

Predict whether a user **Purchased** from **Age** and **EstimatedSalary** using Random Forest.

| Column | Role | Description |
|--------|------|-------------|
| `Age` | Feature (X₁) | User age in years |
| `EstimatedSalary` | Feature (X₂) | Estimated annual salary in USD |
| `Purchased` | Target (y) | 0 = No, 1 = Yes |

**File:** `../Datasets/social_network_ads.csv`

---
# Phase 1: Data Pre-processing

Prepare the data before training — same template is reused for other algorithms.

## Example 1 — Cell 1: Load and Explore Data

Load the CSV file and perform initial exploration (head, info, describe, shape).

**What this cell does:** Reads `../Datasets/social_network_ads.csv` and displays basic statistics.

In [ ]:
# Step 1) Load dataset
dataset = pd.read_csv('../Datasets/social_network_ads.csv')  # Read CSV into a DataFrame

FEATURE_COLS = ['Age', 'EstimatedSalary']  # Two numeric input features
TARGET_COL = 'Purchased'                    # Binary target: 0 = No, 1 = Yes

print('First 5 rows:')
display(dataset.head())

print('\nDataset info:')
dataset.info()

print('\nStatistical summary:')
display(dataset.describe())

print(f'\nClass balance ({TARGET_COL}):')
print(dataset[TARGET_COL].value_counts())

print(f'\nShape: {dataset.shape[0]} rows x {dataset.shape[1]} columns')

## Example 1 — Cell 2: Data Cleaning (Handling Missing Values)

Check missing values, remove duplicates, and apply imputation if needed.

**What this cell does:** Cleans the dataset before modeling.

In [ ]:
# Step 2) Data cleaning

print('Missing values per column:')
print(dataset.isnull().sum())

rows_before = len(dataset)
dataset = dataset.drop_duplicates().reset_index(drop=True)
rows_after = len(dataset)
print(f'\nDuplicates removed: {rows_before - rows_after}')

num_cols = dataset.select_dtypes(include=[np.number]).columns.tolist()
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
if dataset.isnull().sum().sum() > 0:
    dataset[num_cols] = imputer.fit_transform(dataset[num_cols])
    print('Missing values imputed with mean')
else:
    print('No missing values — imputer not applied')

print(f'\nRows after cleaning: {rows_after}')

## Example 1 — Cell 3: Categorical Data Encoding

All columns are numeric — encoding is skipped.

**What this cell does:** Confirms no categorical encoding is needed.

In [ ]:
# Step 3) Categorical encoding

cat_cols = dataset.select_dtypes(include=['object', 'category']).columns.tolist()
if cat_cols:
    print(f'Categorical columns found: {cat_cols}')
else:
    print('No categorical columns — encoding skipped.')
    print(f'Feature columns: {FEATURE_COLS}')
    print(f'Target column: {TARGET_COL} (already 0/1)')

## Example 1 — Cell 4: Splitting the Data

Define X and y, then split 80/20 with **stratify** to preserve class ratio.

**What this cell does:** Creates feature/target arrays and applies stratified train_test_split.

In [ ]:
# Step 4) Train-Test split (stratified for classification)

X = dataset[FEATURE_COLS].values  # Feature matrix: Age and EstimatedSalary
y = dataset[TARGET_COL].values    # Target vector: Purchased (0 or 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')
print(f'y_train class counts: {np.bincount(y_train)}')
print(f'y_test class counts:  {np.bincount(y_test)}')

> **Note:** Random Forest Classification does **not** require feature scaling. Trees split on thresholds — scale does not matter.

---
# Phase 2: Random Forest Classification — Social Network Ads

Train a forest of trees and evaluate by majority vote.

## Example 1 — Cell 5: Train the Model

Train `RandomForestClassifier` with `n_estimators=100` trees.

**What this cell does:** Fits the forest and prints the number of trees.

In [ ]:
# Step 5) Train Random Forest Classifier

classifier = RandomForestClassifier(
    n_estimators=100,      # Number of trees in the forest
    max_depth=8,           # Limit depth per tree
    min_samples_leaf=3,    # Minimum samples required in a leaf node
    max_features='sqrt',   # Random subset of features at each split
    random_state=42,       # Reproducible bootstrap and splits
    n_jobs=-1              # Use all CPU cores for faster training
)

classifier.fit(X_train, y_train)  # Train all trees on bootstrap samples

print('Random Forest trained successfully.')
print(f'Number of trees (estimators): {len(classifier.estimators_)}')  # Should equal n_estimators
print(f'Feature importances: {classifier.feature_importances_.round(4)}')  # Age vs Salary importance

## Example 1 — Cell 6: Predict

Predict class labels by **majority vote** across all trees.

**What this cell does:** Uses `predict()` and `predict_proba()` (vote fractions).

In [ ]:
# Step 6) Predict classes and probabilities

y_pred_train = classifier.predict(X_train)            # Majority vote for training set
y_pred_test = classifier.predict(X_test)              # Majority vote for test set
y_proba_test = classifier.predict_proba(X_test)[:, 1]  # Fraction of trees voting class 1

print('Sample predictions (Test set):')
for i in range(min(5, len(y_test))):
    label = 'Yes' if y_pred_test[i] == 1 else 'No'
    actual = 'Yes' if y_test[i] == 1 else 'No'
    print(f'  Actual={actual}, Predicted={label}, Tree vote P(Yes)={y_proba_test[i]:.2%}')

## Example 1 — Cell 7: Visualization

Plot **decision regions** from the Random Forest (smoother than a single tree).

**What this cell does:** Shows voted classification regions in the Age–Salary plane.

In [ ]:
# Step 7) Visualization — Random Forest decision regions

x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 5000, X[:, 1].max() + 5000
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
grid = np.c_[xx.ravel(), yy.ravel()]
Z = classifier.predict(grid).reshape(xx.shape)  # Majority vote at each grid point

plt.figure(figsize=(10, 6))
plt.contourf(xx, yy, Z, alpha=0.25, cmap='RdYlGn')
plt.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1], c='red', label='Not Purchased', alpha=0.5, s=40)
plt.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1], c='green', label='Purchased', alpha=0.5, s=40)
plt.scatter(X_test[:, 0], X_test[:, 1], c='blue', edgecolors='k', s=80, label='Test points')
plt.xlabel('Age')
plt.ylabel('Estimated Salary (USD)')
plt.title('Random Forest Decision Regions — Social Network Ads')
plt.legend()
plt.tight_layout()
plt.show()

## Example 1 — Cell 8: Evaluation

Evaluate with Accuracy, Precision, Recall, F1, and Confusion Matrix.

**What this cell does:** Computes classification metrics on the test set.

In [ ]:
# Step 8) Evaluation — classification metrics

acc = accuracy_score(y_test, y_pred_test)
prec = precision_score(y_test, y_pred_test, zero_division=0)
rec = recall_score(y_test, y_pred_test, zero_division=0)
f1 = f1_score(y_test, y_pred_test, zero_division=0)

results = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Value': [acc, prec, rec, f1],
    'Description': [
        'Fraction of correct predictions',
        'Of predicted Yes, fraction actually Yes',
        'Of actual Yes, fraction correctly predicted',
        'Balance between Precision and Recall'
    ]
})

display(results.round(4))

cm = confusion_matrix(y_test, y_pred_test)
print('\nConfusion Matrix (rows=Actual, cols=Predicted):')
print(cm)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No (0)', 'Yes (1)']).plot(ax=ax, cmap='Blues')
plt.title('Confusion Matrix — Random Forest')
plt.tight_layout()
plt.show()

print(f'\nExample 1 Test Accuracy = {acc:.4f}')

## Why does Random Forest work for Social Network Ads?

| # | Reason | Explanation |
|---|--------|-------------|
| 1 | **Non-linear patterns** | Purchase behavior varies by Age/Salary — forest captures complex regions |
| 2 | **Reduces overfitting** | Voting across 100 trees is more stable than one deep CART |
| 3 | **No scaling needed** | Trees split on raw Age and Salary thresholds |
| 4 | **Bootstrap diversity** | Each tree sees a slightly different sample — robust ensemble |
| 5 | **Probability from votes** | P(Yes) = fraction of trees voting Yes — useful for ranking leads |

> **Summary:** Random Forest **votes** across many trees — smoother and often more accurate than one CART alone.

## Understanding Classification Metrics — Example 1

| Metric | Random Forest Context |
|--------|----------------------|
| **Accuracy** | Overall correct majority votes |
| **Precision** | When forest predicts Yes, how often correct |
| **Recall** | Of all actual Yes, how many the forest catches |
| **F1** | Balance when classes are imbalanced |

> **Compare:** Random Forest vs single Decision Tree on the same data — RF usually has **higher test accuracy**.

---
# Example 2: Loan Approval — Random Forest Classification

Predict **Approved** vs **Rejected** from four financial features using Random Forest.

| Column | Role | Description |
|--------|------|-------------|
| `CreditScore` | Feature (X₁) | Credit score (300–850) |
| `Income` | Feature (X₂) | Annual income in USD |
| `LoanAmount` | Feature (X₃) | Requested loan amount in USD |
| `YearsEmployed` | Feature (X₄) | Years at current job |
| `Approved` | Target (y) | 1 = Approved, 0 = Rejected |

**File:** `../Datasets/loan_approval.csv`

## Example 2 — Cell 1: Load and Explore Data

Load the loan approval CSV and inspect the data.

**What this cell does:** Reads `../Datasets/loan_approval.csv` and displays basic statistics.

In [ ]:
# Step 1) Load dataset
dataset = pd.read_csv('../Datasets/loan_approval.csv')

FEATURE_COLS = ['CreditScore', 'Income', 'LoanAmount', 'YearsEmployed']
TARGET_COL = 'Approved'

print('First 5 rows:')
display(dataset.head())

print('\nDataset info:')
dataset.info()

print('\nStatistical summary:')
display(dataset.describe())

print(f'\nClass balance ({TARGET_COL}):')
print(dataset[TARGET_COL].value_counts())

print(f'\nShape: {dataset.shape[0]} rows x {dataset.shape[1]} columns')

## Example 2 — Cell 2: Data Cleaning (Handling Missing Values)

This dataset includes missing values in features to demonstrate `SimpleImputer`.

**What this cell does:** Checks for nulls, removes duplicates, and imputes missing values.

In [ ]:
# Step 2) Data cleaning

print('Missing values per column:')
print(dataset.isnull().sum())

rows_before = len(dataset)
dataset = dataset.drop_duplicates().reset_index(drop=True)
print(f'\nDuplicates removed: {rows_before - len(dataset)}')

if dataset[TARGET_COL].isnull().sum() > 0:
    dataset = dataset.dropna(subset=[TARGET_COL]).reset_index(drop=True)
    print('Rows with missing target removed')

imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
cols = FEATURE_COLS + [TARGET_COL]
if dataset[cols].isnull().sum().sum() > 0:
    dataset[cols] = imputer.fit_transform(dataset[cols])
    print('Missing feature values imputed with mean')
else:
    print('No missing values — imputer not applied')

print(f'\nRows after cleaning: {len(dataset)}')

## Example 2 — Cell 3: Categorical Data Encoding

All columns are numeric — encoding is skipped.

**What this cell does:** Confirms no categorical encoding is needed.

In [ ]:
# Step 3) Categorical encoding

cat_cols = dataset.select_dtypes(include=['object', 'category']).columns.tolist()
if cat_cols:
    print(f'Categorical columns found: {cat_cols}')
else:
    print('No categorical columns — encoding skipped.')
    print(f'Feature columns: {FEATURE_COLS}')

## Example 2 — Cell 4: Splitting the Data

Define X (4 features) and y (Approved), then split 80/20 with stratify.

**What this cell does:** Creates feature/target arrays and applies stratified train_test_split.

In [ ]:
# Step 4) Train-Test split

X = dataset[FEATURE_COLS].values
y = dataset[TARGET_COL].values.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')
print(f'y_train class counts: {np.bincount(y_train)}')
print(f'y_test class counts:  {np.bincount(y_test)}')

## Example 2 — Cell 5: Train the Model

Train Random Forest on four financial features — each tree votes on approval.

**What this cell does:** Fits the forest and reports feature importances.

In [ ]:
# Step 5) Train Random Forest Classifier

classifier = RandomForestClassifier(
    n_estimators=100,      # 100 trees in the ensemble
    max_depth=8,           # Allow deeper trees — more data than Example 1
    min_samples_split=4,   # Need at least 4 samples to split a node
    min_samples_leaf=2,    # Each leaf must have at least 2 samples
    max_features='sqrt',   # sqrt(4) ≈ 2 features considered per split
    random_state=42,
    n_jobs=-1              # Parallel training across CPU cores
)

classifier.fit(X_train, y_train)

print('Random Forest trained successfully.')
print(f'Number of trees: {len(classifier.estimators_)}')

print('\nFeature importances:')
for name, imp in zip(FEATURE_COLS, classifier.feature_importances_):
    print(f'  {name:15s} -> {imp:.4f}')

## Example 2 — Cell 6: Predict

Predict loan approval on the test set by majority vote.

**What this cell does:** Generates ensemble predictions.

In [ ]:
# Step 6) Predict

y_pred_test = classifier.predict(X_test)
y_proba_test = classifier.predict_proba(X_test)[:, 1]

print('Sample predictions (Test set):')
for i in range(min(5, len(y_test))):
    actual = 'Approved' if y_test[i] == 1 else 'Rejected'
    pred = 'Approved' if y_pred_test[i] == 1 else 'Rejected'
    print(f'  Actual={actual}, Predicted={pred}, Tree vote P(Approved)={y_proba_test[i]:.2%}')

## Example 2 — Cell 7: Visualization

Plot **feature importances** and **confusion matrix**.

**What this cell does:** Shows which features drive approval decisions across all trees.

In [ ]:
# Step 7) Visualization — feature importances + confusion matrix

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(FEATURE_COLS, classifier.feature_importances_, color='darkgreen')
axes[0].set_xlabel('Importance')
axes[0].set_title('Feature Importances — Loan Approval (Random Forest)')

cm = confusion_matrix(y_test, y_pred_test)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Rejected', 'Approved']).plot(ax=axes[1], cmap='Blues')
axes[1].set_title('Confusion Matrix — Random Forest')

plt.tight_layout()
plt.show()

## Example 2 — Cell 8: Evaluation

Full classification report with Accuracy, Precision, Recall, and F1.

**What this cell does:** Computes and displays evaluation metrics.

In [ ]:
# Step 8) Evaluation

acc = accuracy_score(y_test, y_pred_test)
prec = precision_score(y_test, y_pred_test, zero_division=0)
rec = recall_score(y_test, y_pred_test, zero_division=0)
f1 = f1_score(y_test, y_pred_test, zero_division=0)

results = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Value': [acc, prec, rec, f1],
    'Description': [
        'Fraction of correct predictions',
        'Of predicted Approved, fraction actually approved',
        'Of actual Approved, fraction correctly predicted',
        'Balance between Precision and Recall'
    ]
})

display(results.round(4))

print('\nDetailed classification report:')
print(classification_report(y_test, y_pred_test, target_names=['Rejected (0)', 'Approved (1)']))

print(f'\nExample 2 Test Accuracy = {acc:.4f}')

## Why does Random Forest work well for Loan Approval?

| # | Reason | Explanation |
|---|--------|-------------|
| 1 | **Multiple features** | Forest splits on Credit, Income, Loan, and Years across many trees |
| 2 | **Non-linear rules** | Complex approval logic — ensemble captures interactions |
| 3 | **Feature importances** | Shows CreditScore often dominates (averaged over all trees) |
| 4 | **No scaling needed** | Raw financial values work directly |
| 5 | **High accuracy** | Voting typically beats a single Decision Tree on this dataset |

> **Compare:** Random Forest vs CART — same interpretability trade-off, but RF usually gives **higher accuracy** and **lower overfitting**.

## Understanding Classification Metrics — Example 2

| Scenario | Focus Metric |
|----------|--------------|
| **False approval costly** | **Precision** |
| **False rejection costly** | **Recall** |
| **Balanced cost** | **Accuracy** or **F1** |

> **Overfitting check:** If train accuracy ≈ 100% but test accuracy is lower, reduce `max_depth` or increase `min_samples_leaf`.